In [11]:
import sys
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").is_dir():
    p = p.parent

sys.path.insert(0, str(p))

In [12]:
import pandas as pd
import matplotlib.pyplot as plt

import json
from pathlib import Path

from src.llm_service.pipeline.graph import FinancialAgent
from src.llm_service.pipeline.reasoning_graph import MultiStepFinancialAgent

In [13]:
REPO_ROOT = p
BENCHMARK_QNA = REPO_ROOT / "data" / "mapping" / "financebench_open_source.jsonl"

In [14]:
benchmark_df = pd.read_json(BENCHMARK_QNA, lines=True)
qna_df = benchmark_df[["question", "answer", "evidence"]]

In [15]:
agent = FinancialAgent()
multi_agent = MultiStepFinancialAgent()

In [16]:
idx = 0
question, answer, evidence = qna_df.loc[idx, "question"], qna_df.loc[idx, "answer"], qna_df.loc[idx, "evidence"]
question, answer, evidence

('What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
 '$1577.00',
 [{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncontrolling interest to net cash\nprovided by operating activities\n \n \n \n \n \n \n \nDepreciation and amortization\n \n \n1,488 \n \n1,544 \n \n1,474 \nCompany pension and postretirement contributions\n \n \n(370) \n \n(967) \n \n(383) \nCompany pension and postretirement expense\n \n \n410 \n \n334 \n \n250 \nStock-based compensation expense\n \n \n302 \n \n324 \n \n298 \nGain on sale of businesses\n \n \n(545) \n \n(586) \n \n(111) \nDeferr

In [17]:
# Base agent (single-step)
# result_base = agent.invoke(question)

# Multi-step agent (decompose -> answer sub-questions -> synthesize)
result_multi = multi_agent.invoke(question)
result_multi, answer

status_code = 200
response_text = {
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "What was 3M's capital expenditure in FY2018?"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 839,
    "candidatesTokenCount": 16,
    "totalTokenCount": 855,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 839
      }
    ]
  },
  "modelVersion": "gemini-2.5-flash-lite",
  "responseId": "-hvAaczTPIWgjuMPm5yKuQM"
}



({'query': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
  'route': 'reasoning_graph',
  'source': 'multi_step',
  'answer': 'The answer cannot be fully determined from the provided information.',
  'error': None,
  'original_query': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
  'translated_questions': ["What was 3M's capital expenditure in FY2018?"],
  'step_results': [{'query': "What was 3M's capital expenditure in FY2018?",
    'metadata': QueryMetadata(company_name='3M', year=2018, report_type=None),
    'route': 'knowledge_retrieval',
    'answer': "The provided documents do not contain information about 3M's capital expenditure in FY2018.\n\nSources:\n1. C:\\Users\\admin\\Desktop\\gic\\data\\pdfs\\3M_2018_10K.pdf (page 48)",
    'source': '

In [20]:
import fitz

pdf_path = r"C:\Users\admin\Desktop\gic\data\pdfs\3M_2018_10K.pdf"

doc = fitz.open(pdf_path)

page_idx = 58
page = doc[page_idx]

text = page.get_text()

print(text)

doc.close()

Table of Contents 
3M Company and Subsidiaries
Consolidated Statement of Changes in Equit y
Years Ended December 31
 
  
 
 
3M Company Shareholders
 
 
 
 
 
     
 
    Common      
 
     
 
    
Accumulated
     
 
 
 
  
 
 
Stock and
  
 
  
 
 
Other
 
 
 
 
 
  
 
 Additional   
 
  
 
 
Comprehensive
 
Non-
 
 
  
 
 
Paid-in
 Retained  Treasury  
Income
 
controlling  
(Dollars in millions, except per share amounts)
 
Total
 
Capital
 Earnings  
Stock
 
(Loss)
 
Interest
 
Balance at December 31, 2015
 
$
11,468  
$
4,800  
$
36,296  
$ (23,308) 
$
(6,359) 
$
39  
 
 
 
  
 
  
 
  
 
  
 
  
 
  
Net income
 
 
5,058  
 
  
 
5,050  
 
  
 
  
 
 8  
Other comprehensive income (loss), net of tax:
 
 
  
 
  
 
  
 
  
 
  
 
  
Cumulative translation adjustment
 
 
(331) 
 
  
 
  
 
  
 
(329) 
 
(2) 
Defined benefit pension and post-retirement plans adjustment
 
 
(524) 
 
  
 
  
 
  
 
(524) 
 
 —  
Cash flow hedging instruments - unrealized gain (loss)
 
 
(33) 
 
  
 
